In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("spark-local-app")
        .master("local[*]")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 19:25:06 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/13 19:25:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/13 19:25:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import pandas as pd

orders_df = pd.read_csv("orders.csv", delimiter=",", encoding="utf-8")
orders_df.head(3)

,order_id,order_date,order_customer_id,order_status,order_total,discount_amount,shipping_cost,payment_method,channel,is_returned
0,ORD-1001,2024-01-15,CUST-201,completed,1250.5,120.0,150.0,credit_card,website,False
1,ORD-1002,2024-01-16,CUST-202,pending,890.0,0.0,95.0,paypal,mobile_app,False
2,ORD-1003,2024-01-17,CUST-203,cancelled,340.0,34.0,50.0,credit_card,website,False


In [7]:
orders_df.columns

Index(['order_id', 'order_date', 'order_customer_id', 'order_status',
       'order_total', 'discount_amount', 'shipping_cost', 'payment_method',
       'channel', 'is_returned'],
      dtype='str')

In [11]:
df_orders = spark.createDataFrame(
    orders_df,
    schema="""
    order_id STRING,
    order_date STRING, 
    order_customer_id STRING,
    order_status STRING,
    order_total FLOAT,
    discount_amount FLOAT,
    shipping_cost FLOAT,
    payment_method STRING,
    channel STRING,
    is_returned BOOLEAN
    """
)

In [12]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: float (nullable = true)
 |-- discount_amount: float (nullable = true)
 |-- shipping_cost: float (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- is_returned: boolean (nullable = true)



In [14]:
df_orders.show()

+--------+----------+-----------------+------------+-----------+---------------+-------------+--------------+----------+-----------+
|order_id|order_date|order_customer_id|order_status|order_total|discount_amount|shipping_cost|payment_method|   channel|is_returned|
+--------+----------+-----------------+------------+-----------+---------------+-------------+--------------+----------+-----------+
|ORD-1001|2024-01-15|         CUST-201|   completed|     1250.5|          120.0|        150.0|   credit_card|   website|      false|
|ORD-1002|2024-01-16|         CUST-202|     pending|      890.0|            0.0|         95.0|        paypal|mobile_app|      false|
|ORD-1003|2024-01-17|         CUST-203|   cancelled|      340.0|           34.0|         50.0|   credit_card|   website|      false|
|ORD-1004|2024-01-18|         CUST-204|   completed|    2100.75|          210.0|        200.0| bank_transfer|   website|       true|
|ORD-1005|2024-01-19|         CUST-205|   completed|     560.25|     

In [20]:
from pyspark.sql.functions import date_format

df_orders.select("*", date_format("order_date", "yyyyMM").alias("order_month")).show()

+--------+----------+-----------------+------------+-----------+---------------+-------------+--------------+----------+-----------+-----------+
|order_id|order_date|order_customer_id|order_status|order_total|discount_amount|shipping_cost|payment_method|   channel|is_returned|order_month|
+--------+----------+-----------------+------------+-----------+---------------+-------------+--------------+----------+-----------+-----------+
|ORD-1001|2024-01-15|         CUST-201|   completed|     1250.5|          120.0|        150.0|   credit_card|   website|      false|     202401|
|ORD-1002|2024-01-16|         CUST-202|     pending|      890.0|            0.0|         95.0|        paypal|mobile_app|      false|     202401|
|ORD-1003|2024-01-17|         CUST-203|   cancelled|      340.0|           34.0|         50.0|   credit_card|   website|      false|     202401|
|ORD-1004|2024-01-18|         CUST-204|   completed|    2100.75|          210.0|        200.0| bank_transfer|   website|       tru

In [21]:
df_orders.withColumn("order_month", date_format("order_date", "yyyyMM")).show()

+--------+----------+-----------------+------------+-----------+---------------+-------------+--------------+----------+-----------+-----------+
|order_id|order_date|order_customer_id|order_status|order_total|discount_amount|shipping_cost|payment_method|   channel|is_returned|order_month|
+--------+----------+-----------------+------------+-----------+---------------+-------------+--------------+----------+-----------+-----------+
|ORD-1001|2024-01-15|         CUST-201|   completed|     1250.5|          120.0|        150.0|   credit_card|   website|      false|     202401|
|ORD-1002|2024-01-16|         CUST-202|     pending|      890.0|            0.0|         95.0|        paypal|mobile_app|      false|     202401|
|ORD-1003|2024-01-17|         CUST-203|   cancelled|      340.0|           34.0|         50.0|   credit_card|   website|      false|     202401|
|ORD-1004|2024-01-18|         CUST-204|   completed|    2100.75|          210.0|        200.0| bank_transfer|   website|       tru

In [23]:
(
    df_orders
        .filter(date_format("order_date", "yyyyMM") == 202402)
        .show()
)

+--------+----------+-----------------+------------+-----------+---------------+-------------+--------------+----------+-----------+
|order_id|order_date|order_customer_id|order_status|order_total|discount_amount|shipping_cost|payment_method|   channel|is_returned|
+--------+----------+-----------------+------------+-----------+---------------+-------------+--------------+----------+-----------+
|ORD-1018|2024-02-01|         CUST-218|   completed|      520.0|            0.0|         65.0|   credit_card|   website|      false|
|ORD-1019|2024-02-02|         CUST-219|   completed|    1890.75|          189.0|        190.0| bank_transfer|   website|      false|
|ORD-1020|2024-02-03|         CUST-220|   cancelled|      410.0|           41.0|         55.0|        paypal|mobile_app|      false|
+--------+----------+-----------------+------------+-----------+---------------+-------------+--------------+----------+-----------+



In [25]:
(
    df_orders
        .groupBy(date_format("order_date", "yyyyMM").alias("order_date"))
        .count()
        .show()
)

+----------+-----+
|order_date|count|
+----------+-----+
|    202401|   17|
|    202402|    3|
+----------+-----+



In [26]:
spark.stop()